# UIT DSC 2026 – LegalQA Hybrid RAG (BM25 + Dense FAISS + RRF + Reranker + LLM)

Notebook này triển khai trọn gói giải pháp Hybrid RAG tiên tiến cho cuộc thi UIT DSC 2026:

1. **Tự động nhận diện môi trường**: Kaggle (GPU T4/P100), Google Colab, hoặc Local.
2. **Tự tìm dataset**: `train.json`, `public-official.json` và `selected-contexts` (thư mục lồng `selected-contexts/selected-contexts` hoặc file zip).
3. **Hybrid Retrieval đa tầng**:
   - **BM25 Top-50** qua SQLite FTS5.
   - **Dense FAISS Top-50** qua `AITeamVN/Vietnamese_Embedding_v2`.
   - **RRF Fusion (k=60)** hợp nhất thứ hạng $\rightarrow$ chọn **Top-50 candidate chunks**.
   - **Cross-Encoder Reranker** qua `AITeamVN/Vietnamese_Reranker` $\rightarrow$ chọn **Top-3 chunks** chính xác nhất.
4. **LLM Generation**: Dùng `AITeamVN/Vi-Qwen2-1.5B-RAG` cùng SYSTEM_PROMPT & RAG_TEMPLATE.
5. **Quan sát dataset**: thống kê độ dài, dữ liệu trùng/rỗng và xem mẫu Train/corpus.
6. **Đánh giá**: pseudo Retrieval Recall@1/@3/@5 cho từng tầng; Answer Token-F1, METEOR và ROUGE-L theo scoring program BTC.
7. **Hiển thị tiến trình realtime & tự động lưu checkpoint định kỳ**.
8. **Kiểm tra định dạng submission** và đóng gói `.zip` sẵn sàng nộp lên Codabench.

> **Lưu ý:** Train không có nhãn `context_id/chunk_id`; Retrieval Recall trong notebook dùng pseudo-gold suy ra từ answer và luôn báo coverage.

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import sqlite3
import statistics
import subprocess
import sys
import time
import zipfile
from pathlib import Path

# Set before any subprocess imports Torch; helps avoid CUDA allocator fragmentation.
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

from IPython.display import FileLink, Markdown, display

# ===== CẤU HÌNH CHÍNH =====
REPO_URL = "https://github.com/lighth-gh/uit-dsc-2026-task2-legalqa.git"

# Chọn workspace theo thư mục runtime thực tế; không dựa vào module/biến môi trường.
if Path("/kaggle/working").is_dir():
    RUNTIME_PLATFORM = "Kaggle"
    REPO_DIR = Path("/kaggle/working/uit-dsc-2026-task2-legalqa")
    INPUT_ROOT = Path("/kaggle/input")
    WORK_DIR = Path("/kaggle/working/legalqa-run")
elif Path("/content").is_dir():
    RUNTIME_PLATFORM = "Colab"
    REPO_DIR = Path("/content/uit-dsc-2026-task2-legalqa")
    INPUT_ROOT = Path("/content")
    WORK_DIR = Path("/content/legalqa-run")
else:
    RUNTIME_PLATFORM = "Local"
    # Môi trường Local máy tính
    REPO_DIR = Path("./UIT_DSC_2026_LegalQA_baseline_v0.1").resolve()
    if not REPO_DIR.exists():
        REPO_DIR = Path(".").resolve()
    INPUT_ROOT = Path(".").resolve()
    WORK_DIR = REPO_DIR / "artifacts"

# Mọi file lâu dài nằm trong WORK_DIR; Kaggle sẽ thu thập /kaggle/working.
EXPORT_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else WORK_DIR
MODEL_STORE_DIR = WORK_DIR / "models"
HF_CACHE_DIR = WORK_DIR / "hf-cache"
os.environ["HF_HOME"] = str(HF_CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE_DIR / "hub")

# Ghi đè đường dẫn thủ công nếu cần (để None để tự động quét):
DATASET_DIR = None
TRAIN_PATH = None
PUBLIC_PATH = None
CONTEXTS_PATH = None

# Cấu hình thực thi index
FORCE_REBUILD_INDEX = False
BUILD_DENSE_INDEX = True      # Dense là thành phần bắt buộc của cấu hình RAG mặc định
RUN_TESTS = True
RUN_RETRIEVAL_EVAL = False   # Bật khi cần báo cáo; tắt để đi thẳng tới submission
RETRIEVAL_EVAL_LIMIT = 100
RUN_VALIDATION = False       # Bật khi cần metric; validation 100 mẫu tốn khoảng 30 phút
VALIDATION_LIMIT = 100       # 100 nhanh hơn; full baseline dùng 300
OFFICIAL_METRICS = True

# Chế độ sinh: "rag", "hybrid_rag", "hybrid", "extractive", "knn"
MODE = "rag"                 # Khuyên dùng "rag" hoặc "hybrid_rag"
KNN_THRESHOLD = 0.72

# Cấu hình Hybrid Retrieval & Reranking
BM25_TOP_K = 50
DENSE_TOP_K = 50
RRF_K = 60
RRF_TOP_K = 50
RERANK_TOP_K = 3             # Top 3 chunks đưa vào LLM
DENSE_QUERY_MAX_LENGTH = 256
DENSE_DOCUMENT_MAX_LENGTH = 2048
DENSE_BATCH_SIZE = 8           # Total batch; encoder splits it across all visible GPUs
DENSE_CHECKPOINT_CHUNKS = 4096 # Lưu part dense khoảng mỗi 2-3 phút
RERANKER_MAX_LENGTH = 2304
ALLOW_RETRIEVAL_FALLBACK = False
EMBEDDING_MODEL_ID = "AITeamVN/Vietnamese_Embedding_v2"
RERANKER_MODEL_ID = "AITeamVN/Vietnamese_Reranker"
GENERATOR_MODEL_ID = "AITeamVN/Vi-Qwen2-1.5B-RAG"
EMBEDDING_MODEL = EMBEDDING_MODEL_ID
RERANKER_MODEL = RERANKER_MODEL_ID
GENERATOR_MODEL = GENERATOR_MODEL_ID
SAVE_MODEL_SNAPSHOTS = True   # Lưu model vào Kaggle Output để tái sử dụng

# Cấu hình LLM Generator
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.0
TOP_P = 0.9
MAX_INPUT_TOKENS = 7168
GENERATION_SEED = 2026
DEVICE = "auto"           # Tự kiểm tra CUDA thay vì ép GPU theo môi trường
CHECKPOINT_INTERVAL = 1      # Ghi partial submission + checkpoint sau từng câu

WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"Môi trường: {RUNTIME_PLATFORM}")
print(f"Python: {sys.version.split()[0]}")
print(f"Repo dir: {REPO_DIR}")
print(f"Working directory: {WORK_DIR}")
print(f"Persistent export: {EXPORT_DIR}")
print(f"Pipeline: BM25({BM25_TOP_K}) + Dense({DENSE_TOP_K}) -> RRF({RRF_K}) Top-{RRF_TOP_K} -> Reranker Top-{RERANK_TOP_K} -> {GENERATOR_MODEL}")


## 1. Clone hoặc cập nhật mã nguồn

In [ ]:
def run(command: list[str], cwd: Path | None = None) -> None:
    """Chạy lệnh terminal và stream log realtime ra output cell."""
    print("$", " ".join(map(str, command)))
    process = subprocess.Popen(
        command,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        encoding="utf-8",
        errors="replace",
    )
    if process.stdout is not None:
        for line in iter(process.stdout.readline, ""):
            print(line, end="", flush=True)
        process.stdout.close()
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)


if RUNTIME_PLATFORM != "Local":
    if (REPO_DIR / ".git").is_dir():
        run(["git", "pull", "--ff-only", "origin", "main"], cwd=REPO_DIR)
    else:
        if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
            import shutil
            shutil.rmtree(REPO_DIR, ignore_errors=True)
        run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])

try:
    commit = subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"],
        cwd=REPO_DIR,
        text=True,
    ).strip()
    print(f"Mã nguồn tại commit: {commit} ({REPO_DIR})")
except Exception:
    print(f"Đang dùng local folder: {REPO_DIR}")


## 2. Cài đặt thư viện Deep Learning, FAISS & Reranker

In [ ]:
if "rag" in MODE or RUN_RETRIEVAL_EVAL:
    req_path = REPO_DIR / "requirements-generator.txt"
    if req_path.exists():
        run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req_path)])
        print("Đã cài đặt requirements-generator.txt")
    else:
        run([sys.executable, "-m", "pip", "install", "-q", "torch", "transformers>=4.40.0", "accelerate", "sentencepiece", "tqdm", "faiss-cpu", "numpy"])
        print("Đã cài đặt torch, transformers, faiss, reranker dependencies")
    import torch
    if torch.cuda.is_available():
        print(f"CUDA ready: {torch.cuda.device_count()} visible GPU(s)")
        for gpu_id in range(torch.cuda.device_count()):
            print(f"  GPU {gpu_id}: {torch.cuda.get_device_name(gpu_id)}")
    else:
        print("WARNING: CUDA is unavailable; dense encoding will run on CPU.")

MODEL_MARKER = ".legalqa_model.json"


def model_snapshot_is_complete(model_dir: Path) -> bool:
    has_weights = (
        any(model_dir.glob("*.safetensors"))
        or any(model_dir.glob("pytorch_model*.bin"))
    )
    return (model_dir / "config.json").is_file() and has_weights


def discover_input_models() -> dict[str, Path]:
    discovered: dict[str, Path] = {}
    if not INPUT_ROOT.exists():
        return discovered
    for marker in INPUT_ROOT.rglob(MODEL_MARKER):
        try:
            payload = json.loads(marker.read_text(encoding="utf-8"))
        except (OSError, ValueError):
            continue
        repo_id = payload.get("repo_id") if isinstance(payload, dict) else None
        if isinstance(repo_id, str) and model_snapshot_is_complete(marker.parent):
            discovered.setdefault(repo_id, marker.parent)
    return discovered


def prepare_model_snapshot(repo_id: str, input_models: dict[str, Path]) -> str:
    cached_input = input_models.get(repo_id)
    if cached_input is not None:
        print(f"Tái sử dụng model từ Kaggle Input: {repo_id} -> {cached_input}")
        return str(cached_input)
    target = MODEL_STORE_DIR / repo_id.replace("/", "--")
    marker = target / MODEL_MARKER
    if marker.is_file() and model_snapshot_is_complete(target):
        print(f"Tái sử dụng model trong working output: {target}")
        return str(target)
    from huggingface_hub import HfApi, snapshot_download
    revision = HfApi().model_info(repo_id).sha
    if not revision:
        raise RuntimeError(f"Không xác định được revision cho model {repo_id}")
    revision = str(revision)
    print(f"Đang lưu snapshot model: {repo_id}@{revision[:12]} -> {target}")
    snapshot_download(
        repo_id=repo_id,
        revision=revision,
        local_dir=str(target),
        ignore_patterns=["onnx/*", "onnx/**", "*.onnx"],
    )
    if not model_snapshot_is_complete(target):
        raise RuntimeError(f"Snapshot model chưa đủ config/weights: {target}")
    marker_tmp = marker.with_suffix(marker.suffix + ".tmp")
    marker_tmp.write_text(
        json.dumps({"repo_id": repo_id, "revision": revision}, ensure_ascii=False),
        encoding="utf-8",
    )
    marker_tmp.replace(marker)
    return str(target)


if SAVE_MODEL_SNAPSHOTS and "rag" in MODE:
    MODEL_STORE_DIR.mkdir(parents=True, exist_ok=True)
    input_models = discover_input_models()
    EMBEDDING_MODEL = prepare_model_snapshot(EMBEDDING_MODEL_ID, input_models)
    RERANKER_MODEL = prepare_model_snapshot(RERANKER_MODEL_ID, input_models)
    GENERATOR_MODEL = prepare_model_snapshot(GENERATOR_MODEL_ID, input_models)
    print("Model snapshots ready; các thư mục này sẽ xuất hiện trong Kaggle Output.")

if RUN_VALIDATION and OFFICIAL_METRICS:
    metrics_req = REPO_DIR / "requirements-metrics.txt"
    run([sys.executable, "-m", "pip", "install", "-q", "-r", str(metrics_req)])
    run([sys.executable, "-m", "nltk.downloader", "-q", "wordnet", "omw-1.4"])
    print("Đã cài metric BTC và dữ liệu NLTK WordNet")


## 3. Tự động nhận diện dữ liệu đầu vào (Hỗ trợ thư mục lồng nhau)

In [ ]:
def resolve_file(
    label: str,
    override: str | Path | None,
    accepted_names: set[str],
) -> Path:
    if override is not None:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(f"{label}: không tồn tại: {path}")
        return path

    search_roots = [
        Path(DATASET_DIR) if DATASET_DIR is not None else None,
        INPUT_ROOT,
        REPO_DIR,
        REPO_DIR.parent,
        Path("./data"),
        Path("."),
    ]
    accepted = {name.casefold() for name in accepted_names}
    for root in search_roots:
        if root is None or not root.exists():
            continue
        matches = sorted(
            path for path in root.rglob("*")
            if path.is_file() and path.name.casefold() in accepted
        )
        if matches:
            return matches[0]

    raise FileNotFoundError(
        f"Không tìm thấy {label}. Tên chấp nhận: {sorted(accepted_names)}"
    )


def resolve_contexts(override: str | Path | None) -> Path:
    if override is not None:
        p = Path(override)
        if p.exists():
            return p

    search_roots = [
        Path(DATASET_DIR) if DATASET_DIR is not None else None,
        INPUT_ROOT,
        REPO_DIR,
        REPO_DIR.parent,
        Path("./data"),
        Path("."),
    ]
    for root in search_roots:
        if root is None or not root.exists():
            continue
        # 1. Tìm thư mục lồng selected-contexts/selected-contexts hoặc thư mục chứa json
        for p in root.rglob("*selected-contexts*"):
            if p.is_dir():
                # Nếu có thư mục con lồng nhau
                nested = p / "selected-contexts"
                if nested.is_dir() and any(nested.glob("context_*.json")):
                    return nested
                if any(p.glob("context_*.json")) or any(p.rglob("context_*.json")):
                    return p
        # 2. Tìm file zip
        for p in root.rglob("*selected-contexts*.zip"):
            if p.is_file():
                return p
    raise FileNotFoundError("Không tìm thấy selected-contexts (zip hoặc thư mục).")


TRAIN_PATH = resolve_file("Train", TRAIN_PATH, {"train.json"})
PUBLIC_PATH = resolve_file(
    "Public",
    PUBLIC_PATH,
    {"public-official(1).json", "public-official.json", "public_official.json", "public_test.json"},
)
CONTEXTS_PATH = resolve_contexts(CONTEXTS_PATH)

for label, path in [
    ("Train", TRAIN_PATH),
    ("Public", PUBLIC_PATH),
    ("Contexts", CONTEXTS_PATH),
]:
    size_mb = path.stat().st_size / 1024**2 if path.is_file() else 0
    print(f"{label:8s}: {path} ({size_mb:.2f} MiB)")

with TRAIN_PATH.open(encoding="utf-8") as handle:
    train_data = json.load(handle)
with PUBLIC_PATH.open(encoding="utf-8") as handle:
    public_data = json.load(handle)

assert isinstance(train_data, dict) and len(train_data) > 0
assert isinstance(public_data, dict) and len(public_data) > 0
assert all(isinstance(item.get("question"), str) for item in train_data.values())
assert all(isinstance(item.get("question"), str) for item in public_data.values())
print(f"Schema OK — Train: {len(train_data):,}, Public: {len(public_data):,}")


## 4. Quan sát nhanh dataset

In [ ]:
def length_summary(texts: list[str]) -> dict[str, float]:
    lengths = sorted(len(str(text).split()) for text in texts)
    return {
        "min": lengths[0],
        "median": statistics.median(lengths),
        "p90": lengths[round(0.9 * (len(lengths) - 1))],
        "max": lengths[-1],
    }

train_questions = [str(item.get("question") or "") for item in train_data.values()]
train_answers = [str(item.get("answer") or "") for item in train_data.values()]
public_questions = [str(item.get("question") or "") for item in public_data.values()]
dataset_report = {
    "train_samples": len(train_data),
    "public_samples": len(public_data),
    "empty_train_answers": sum(not answer.strip() for answer in train_answers),
    "duplicate_train_questions": len(train_questions) - len(set(train_questions)),
    "train_question_words": length_summary(train_questions),
    "train_answer_words": length_summary(train_answers),
    "public_question_words": length_summary(public_questions),
}
print(json.dumps(dataset_report, ensure_ascii=False, indent=2))

print("\n=== Một vài mẫu Train ===")
for sample_id in list(train_data)[:3]:
    item = train_data[sample_id]
    print(f"\n[{sample_id}] Q: {item['question']}")
    print("A:", " ".join(str(item.get('answer') or '').split()[:120]))

context_examples = []
if CONTEXTS_PATH.is_dir():
    for context_path in sorted(CONTEXTS_PATH.rglob("context_*.json"))[:3]:
        context_examples.append(json.loads(context_path.read_text(encoding="utf-8")))
elif zipfile.is_zipfile(CONTEXTS_PATH):
    with zipfile.ZipFile(CONTEXTS_PATH) as archive:
        for name in sorted(n for n in archive.namelist() if n.endswith(".json"))[:3]:
            context_examples.append(json.loads(archive.read(name).decode("utf-8")))

print("\n=== Một vài văn bản corpus ===")
for context in context_examples:
    passage = str(context.get("passage") or "")
    print(f"\nID={context.get('id')} | {context.get('name')} | {len(passage.split()):,} từ")
    print("Preview:", " ".join(passage.split()[:80]))


## 5. Kiểm tra môi trường và chạy Unit Tests

In [ ]:
connection = sqlite3.connect(":memory:")
try:
    connection.execute("CREATE VIRTUAL TABLE fts_check USING fts5(text)")
finally:
    connection.close()
print("SQLite FTS5: OK")

if RUN_TESTS:
    run(
        [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
        cwd=REPO_DIR,
    )
else:
    print("Bỏ qua unit test theo cấu hình.")


## 6. Dựng BM25 index & Dense Vector FAISS Index

In [ ]:
DB_PATH = WORK_DIR / "legalqa.sqlite"
DENSE_INDEX_PATH = WORK_DIR / "legalqa_dense"


def cached_input_rank(path: Path) -> tuple[int, int, str]:
    # Ưu tiên đúng output của notebook này; tránh nhặt nhầm submission/index mẫu trong dataset.
    has_manifest = any(
        (parent / "legalqa_artifacts.json").is_file()
        for parent in path.parents
    )
    in_run_dir = "legalqa-run" in path.parts
    priority = 0 if has_manifest else (1 if in_run_dir else 2)
    return priority, len(path.parts), str(path)


def find_cached_input(name: str) -> Path | None:
    if not INPUT_ROOT.exists():
        return None
    matches = sorted(
        (path for path in INPUT_ROOT.rglob(name) if path.is_file()),
        key=cached_input_rank,
    )
    return matches[0] if matches else None


def index_is_ready(path: Path) -> bool:
    if not path.is_file():
        return False
    try:
        connection = sqlite3.connect(f"file:{path.resolve()}?mode=ro", uri=True)
        metadata = dict(connection.execute("SELECT key, value FROM metadata"))
        connection.close()
        return int(metadata.get("chunks", 0)) > 0 and int(metadata.get("train_samples", 0)) > 0
    except (sqlite3.Error, ValueError):
        return False


if not FORCE_REBUILD_INDEX and not index_is_ready(DB_PATH):
    cached_db = find_cached_input("legalqa.sqlite")
    if cached_db is not None and index_is_ready(cached_db):
        DB_PATH = cached_db
        print(f"Tái sử dụng BM25 index từ Kaggle Input: {DB_PATH}")


# 1. Dựng BM25 FTS5 index
ready = index_is_ready(DB_PATH)
if FORCE_REBUILD_INDEX or not ready:
    if DB_PATH.exists() and not ready:
        for suffix in ("", "-wal", "-shm"):
            candidate = Path(str(DB_PATH) + suffix)
            if candidate.exists() or candidate.is_symlink():
                candidate.unlink()
        print("Đã dọn index không hoàn chỉnh để xây lại sạch.")
    command = [
        sys.executable, "-m", "legalqa_baseline", "build-index",
        "--contexts", str(CONTEXTS_PATH),
        "--train", str(TRAIN_PATH),
        "--db", str(DB_PATH),
    ]
    if DB_PATH.exists():
        command.append("--force")
    run(command, cwd=REPO_DIR)
else:
    print(f"Tái sử dụng BM25 index hoàn chỉnh: {DB_PATH}")

# 2. Dựng Dense FAISS index (nếu bật BUILD_DENSE_INDEX)
REQUIRED_DENSE_SCHEMA = 4


def model_identity(model_name_or_path: str) -> str:
    marker_path = Path(model_name_or_path) / MODEL_MARKER
    if marker_path.is_file():
        try:
            marker_payload = json.loads(marker_path.read_text(encoding="utf-8"))
            repo_id = marker_payload.get("repo_id")
            revision = marker_payload.get("revision")
            if isinstance(repo_id, str) and repo_id.strip():
                if isinstance(revision, str) and revision.strip():
                    return f"{repo_id.strip()}@{revision.strip()}"
                return repo_id.strip()
        except (OSError, ValueError, TypeError):
            pass
    return str(model_name_or_path)


EXPECTED_EMBEDDING_IDENTITY = model_identity(EMBEDDING_MODEL)


def dense_index_is_ready(meta_path: Path, faiss_path: Path, numpy_path: Path) -> bool:
    if not meta_path.is_file() or not (faiss_path.is_file() or numpy_path.is_file()):
        return False
    try:
        payload = json.loads(meta_path.read_text(encoding="utf-8"))
        manifest = payload.get("manifest", {}) if isinstance(payload, dict) else {}
        return (
            int(manifest.get("schema_version", 0)) >= REQUIRED_DENSE_SCHEMA
            and manifest.get("pooling") == "cls"
            and manifest.get("normalization") == "l2"
            and manifest.get("similarity") == "dot_product"
            and manifest.get("embedding_model") == EXPECTED_EMBEDDING_IDENTITY
        )
    except (OSError, ValueError, TypeError):
        return False


meta_dense = DENSE_INDEX_PATH.with_suffix(".meta.json")
dense_vectors = DENSE_INDEX_PATH.with_suffix(".faiss")
dense_numpy = DENSE_INDEX_PATH.with_suffix(".npy")
dense_ready = dense_index_is_ready(meta_dense, dense_vectors, dense_numpy)
if not FORCE_REBUILD_INDEX and not dense_ready:
    cached_meta = find_cached_input("legalqa_dense.meta.json")
    if cached_meta is not None:
        cached_base = cached_meta.with_name("legalqa_dense")
        cached_faiss = cached_base.with_suffix(".faiss")
        cached_numpy = cached_base.with_suffix(".npy")
        if dense_index_is_ready(cached_meta, cached_faiss, cached_numpy):
            DENSE_INDEX_PATH = cached_base
            meta_dense = cached_meta
            dense_vectors = cached_faiss
            dense_numpy = cached_numpy
            dense_ready = True
            print(f"Tái sử dụng Dense index từ Kaggle Input: {DENSE_INDEX_PATH}")

# Nếu lần trước dừng giữa dense build, sao chép các part nhỏ để --resume tiếp tục.
local_dense_checkpoint = WORK_DIR / "legalqa_dense.dense-checkpoint"
if not dense_ready and not local_dense_checkpoint.exists() and INPUT_ROOT.exists():
    cached_checkpoint_dirs = sorted(
        (path for path in INPUT_ROOT.rglob("legalqa_dense.dense-checkpoint") if path.is_dir()),
        key=cached_input_rank,
    )
    if cached_checkpoint_dirs:
        shutil.copytree(cached_checkpoint_dirs[0], local_dense_checkpoint)
        print(f"Đã phục hồi dense checkpoint: {cached_checkpoint_dirs[0]}")
if BUILD_DENSE_INDEX and (not dense_ready or FORCE_REBUILD_INDEX):
    dense_cmd = [
        sys.executable, "-m", "legalqa_baseline", "build-dense-index",
        "--contexts", str(CONTEXTS_PATH),
        "--dense-index", str(DENSE_INDEX_PATH),
        "--embedding-model", EMBEDDING_MODEL,
        "--embedding-max-length", str(DENSE_DOCUMENT_MAX_LENGTH),
        "--batch-size", str(DENSE_BATCH_SIZE),
        "--device", DEVICE,
        "--resume",
        "--checkpoint-chunks", str(DENSE_CHECKPOINT_CHUNKS),
    ]
    if FORCE_REBUILD_INDEX or any(p.exists() for p in (meta_dense, dense_vectors, dense_numpy)):
        dense_cmd.append("--force")
    run(dense_cmd, cwd=REPO_DIR)
elif dense_ready:
    print(f"Đã tìm thấy Dense Vector Index có sẵn: {DENSE_INDEX_PATH}")
else:
    raise FileNotFoundError("Dense index chưa sẵn sàng; bật BUILD_DENSE_INDEX hoặc cung cấp đủ metadata + vector file.")

# Build chạy ở subprocess nên phải cập nhật lại trạng thái cho các cell sau.
dense_ready = dense_index_is_ready(meta_dense, dense_vectors, dense_numpy)
if BUILD_DENSE_INDEX and not dense_ready:
    raise RuntimeError("Dense build kết thúc nhưng không tìm thấy metadata/vector artifact.")
print(f"Dense ready sau bước build: {dense_ready}")


## 7. Đánh giá Retrieval Recall@1/@3/@5

In [ ]:
RETRIEVAL_EVAL_PATH = WORK_DIR / f"retrieval_eval_{RETRIEVAL_EVAL_LIMIT}.json"
if RUN_RETRIEVAL_EVAL:
    retrieval_cmd = [
        sys.executable, "-m", "legalqa_baseline", "evaluate-retrieval",
        "--train", str(TRAIN_PATH),
        "--db", str(DB_PATH),
        "--output", str(RETRIEVAL_EVAL_PATH),
        "--limit", str(RETRIEVAL_EVAL_LIMIT),
        "--ks", "1,3,5",
        "--dense-index", str(DENSE_INDEX_PATH),
        "--embedding-model", EMBEDDING_MODEL,
        "--reranker-model", RERANKER_MODEL,
        "--device", DEVICE,
        "--bm25-top-k", str(BM25_TOP_K),
        "--dense-top-k", str(DENSE_TOP_K),
        "--rrf-k", str(RRF_K),
        "--rrf-top-k", str(RRF_TOP_K),
        "--dense-query-max-length", str(DENSE_QUERY_MAX_LENGTH),
        "--reranker-max-length", str(RERANKER_MAX_LENGTH),
    ]
    run(retrieval_cmd, cwd=REPO_DIR)
    retrieval_report = json.loads(RETRIEVAL_EVAL_PATH.read_text(encoding="utf-8"))
    print("Pseudo-gold coverage:", f"{retrieval_report['pseudo_gold_coverage']:.1%}")
    rows = ["| Stage | Recall@1 | Recall@3 | Recall@5 | MRR@5 | NDCG@5 | MAP@5 |", "|---|---:|---:|---:|---:|---:|---:|"]
    for stage, values in retrieval_report["metrics"].items():
        rows.append(
            f"| {stage} | {values.get('recall@1', 0.0):.4f} | {values.get('recall@3', 0.0):.4f} | {values.get('recall@5', 0.0):.4f} | {values.get('mrr@5', 0.0):.4f} | {values.get('ndcg@5', 0.0):.4f} | {values.get('map@5', 0.0):.4f} |"
        )
    display(Markdown("\n".join(rows)))
    display(retrieval_report["diagnostics"][:3])
else:
    print("Bỏ qua retrieval evaluation theo cấu hình.")


## 8. Validation Answer Similarity / Competition Metrics

In [ ]:
VALIDATION_PATH = WORK_DIR / f"validation_{VALIDATION_LIMIT}.json"
if RUN_VALIDATION:
    val_cmd = [
        sys.executable, "-m", "legalqa_baseline", "validate",
        "--train", str(TRAIN_PATH),
        "--db", str(DB_PATH),
        "--output", str(VALIDATION_PATH),
        "--limit", str(VALIDATION_LIMIT),
        "--modes", MODE,
        "--knn-threshold", str(KNN_THRESHOLD),
        "--bm25-top-k", str(BM25_TOP_K),
        "--dense-top-k", str(DENSE_TOP_K),
        "--rrf-k", str(RRF_K),
        "--rrf-top-k", str(RRF_TOP_K),
        "--rerank-top-k", str(RERANK_TOP_K),
        "--dense-query-max-length", str(DENSE_QUERY_MAX_LENGTH),
        "--reranker-max-length", str(RERANKER_MAX_LENGTH),
        "--embedding-model", str(EMBEDDING_MODEL),
        "--reranker-model", str(RERANKER_MODEL),
        "--generator-model", str(GENERATOR_MODEL),
        "--device", str(DEVICE),
    ]
    if dense_ready:
        val_cmd.extend(["--dense-index", str(DENSE_INDEX_PATH)])
    if ALLOW_RETRIEVAL_FALLBACK:
        val_cmd.append("--allow-retrieval-fallback")
    if OFFICIAL_METRICS:
        val_cmd.append("--official-metrics")
    run(val_cmd, cwd=REPO_DIR)
    val_data = json.loads(VALIDATION_PATH.read_text(encoding="utf-8"))
    val_rows = ["| Mode | METEOR (approx) | ROUGE-L | Token F1 | BLEU-4 | Exact Match | METEOR (BTC) | ROUGE-L (BTC) |", "|---|---:|---:|---:|---:|---:|---:|---:|"]
    for m, scores in val_data.get("results", {}).items():
        val_rows.append(f"| {m} | {scores.get('meteor_exact_approx', 0.0):.4f} | {scores.get('rougeL', 0.0):.4f} | {scores.get('answer_token_f1', 0.0):.4f} | {scores.get('bleu_4', 0.0):.4f} | {scores.get('exact_match', 0.0):.4f} | {scores.get('competition_meteor', 0.0):.4f} | {scores.get('competition_rougeL', 0.0):.4f} |")
    display(Markdown("\n".join(val_rows)))
    display(val_data)
else:
    print("Validation đang tắt — tiếp tục sinh Public submission.")


## 9. Sinh câu trả lời cho Public Test (Hybrid RAG + Realtime Progress)

In [ ]:
SUBMISSION_PATH = WORK_DIR / f"submission_{MODE}.json"
SUBMISSION_CHECKPOINT_PATH = SUBMISSION_PATH.with_suffix(".checkpoint.json")

# Phục hồi prediction/checkpoint từ output Kaggle cũ nếu đã add làm Input.
for cached_name, destination in [
    (SUBMISSION_CHECKPOINT_PATH.name, SUBMISSION_CHECKPOINT_PATH),
    (SUBMISSION_PATH.name, SUBMISSION_PATH),
]:
    if not destination.exists():
        cached_progress = find_cached_input(cached_name)
        if cached_progress is not None:
            shutil.copy2(cached_progress, destination)
            print(f"Đã phục hồi tiến độ: {cached_progress} -> {destination}")

predict_cmd = [
    sys.executable, "-m", "legalqa_baseline", "predict",
    "--input", str(PUBLIC_PATH),
    "--db", str(DB_PATH),
    "--output", str(SUBMISSION_PATH),
    "--mode", MODE,
    "--knn-threshold", str(KNN_THRESHOLD),
    "--bm25-top-k", str(BM25_TOP_K),
    "--dense-top-k", str(DENSE_TOP_K),
    "--rrf-k", str(RRF_K),
    "--rrf-top-k", str(RRF_TOP_K),
    "--rerank-top-k", str(RERANK_TOP_K),
    "--dense-query-max-length", str(DENSE_QUERY_MAX_LENGTH),
    "--reranker-max-length", str(RERANKER_MAX_LENGTH),
    "--embedding-model", str(EMBEDDING_MODEL),
    "--reranker-model", str(RERANKER_MODEL),
    "--generator-model", str(GENERATOR_MODEL),
    "--device", str(DEVICE),
    "--max-new-tokens", str(MAX_NEW_TOKENS),
    "--temperature", str(TEMPERATURE),
    "--top-p", str(TOP_P),
    "--max-input-tokens", str(MAX_INPUT_TOKENS),
    "--generation-seed", str(GENERATION_SEED),
    "--resume",
    "--checkpoint-interval", str(CHECKPOINT_INTERVAL),
]

if dense_ready:
    predict_cmd.extend(["--dense-index", str(DENSE_INDEX_PATH)])
if ALLOW_RETRIEVAL_FALLBACK:
    predict_cmd.append("--allow-retrieval-fallback")

run(predict_cmd, cwd=REPO_DIR)
assert SUBMISSION_PATH.is_file(), f"Không tìm thấy file submission: {SUBMISSION_PATH}"
print(f"[✓] Đã tạo thành công submission: {SUBMISSION_PATH}")


## 10. Kiểm tra chất lượng và định dạng trước khi nộp

In [ ]:
prediction_data = json.loads(SUBMISSION_PATH.read_text(encoding="utf-8"))

assert set(prediction_data) == set(public_data), (
    f"ID mismatch: public={len(public_data)}, prediction={len(prediction_data)}"
)
assert all(
    isinstance(item, dict) and set(item) == {"answer"}
    for item in prediction_data.values()
), "Mỗi prediction phải có đúng một trường answer"
answers = [item["answer"] for item in prediction_data.values()]
assert all(isinstance(answer, str) and answer.strip() for answer in answers), (
    "Phát hiện answer rỗng/null"
)
lengths = sorted(len(answer.split()) for answer in answers)
quality_report = {
    "ids": len(prediction_data),
    "empty_answers": sum(not answer.strip() for answer in answers),
    "words_min": min(lengths),
    "words_median": statistics.median(lengths),
    "words_p90": lengths[round(0.9 * (len(lengths) - 1))],
    "words_max": max(lengths),
    "file_mib": round(SUBMISSION_PATH.stat().st_size / 1024**2, 2),
}
print(json.dumps(quality_report, ensure_ascii=False, indent=2))
print("Submission schema: OK")


## 11. Xem thử kết quả và tải submission

In [ ]:
for sample_id in list(public_data)[:3]:
    question = public_data[sample_id]["question"]
    answer = prediction_data[sample_id]["answer"]
    preview = answer[:1500] + ("…" if len(answer) > 1500 else "")
    display(Markdown(f"### ID {sample_id}\n**Câu hỏi:** {question}\n\n**Trả lời:** {preview}"))

FINAL_SUBMISSION_PATH = EXPORT_DIR / SUBMISSION_PATH.name
if FINAL_SUBMISSION_PATH.resolve() != SUBMISSION_PATH.resolve():
    shutil.copy2(SUBMISSION_PATH, FINAL_SUBMISSION_PATH)

FINAL_CHECKPOINT_PATH = EXPORT_DIR / SUBMISSION_CHECKPOINT_PATH.name
if SUBMISSION_CHECKPOINT_PATH.is_file() and FINAL_CHECKPOINT_PATH.resolve() != SUBMISSION_CHECKPOINT_PATH.resolve():
    shutil.copy2(SUBMISSION_CHECKPOINT_PATH, FINAL_CHECKPOINT_PATH)

SUBMISSION_ZIP = EXPORT_DIR / f"submission_{MODE}.zip"
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(FINAL_SUBMISSION_PATH, arcname=FINAL_SUBMISSION_PATH.name)

ARTIFACT_MANIFEST_PATH = EXPORT_DIR / "legalqa_artifacts.json"
artifact_manifest = {
    "schema_version": 1,
    "code_revision": globals().get("commit", "unknown"),
    "runtime": RUNTIME_PLATFORM,
    "work_dir": str(WORK_DIR),
    "submission": str(FINAL_SUBMISSION_PATH),
    "submission_zip": str(SUBMISSION_ZIP),
    "prediction_checkpoint": str(FINAL_CHECKPOINT_PATH),
    "bm25_index": str(DB_PATH),
    "dense_index": str(DENSE_INDEX_PATH),
    "models": {
        EMBEDDING_MODEL_ID: str(EMBEDDING_MODEL),
        RERANKER_MODEL_ID: str(RERANKER_MODEL),
        GENERATOR_MODEL_ID: str(GENERATOR_MODEL),
    },
    "reuse": "Save Version trên Kaggle, rồi Add Data/output đó làm Input cho lần chạy sau.",
}
ARTIFACT_MANIFEST_PATH.write_text(
    json.dumps(artifact_manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("Đã xuất JSON/ZIP/checkpoint vào thư mục persistent của Kaggle.")
print(json.dumps(artifact_manifest, ensure_ascii=False, indent=2))
display(FileLink(str(FINAL_SUBMISSION_PATH)))
display(FileLink(str(SUBMISSION_ZIP)))
display(FileLink(str(ARTIFACT_MANIFEST_PATH)))


## Hoàn tất

Tệp cuối, checkpoint, index và model cache đều nằm dưới `/kaggle/working` khi chạy Kaggle. Sau khi job xong hãy chọn **Save Version**; lần chạy sau chọn **Add Data** và thêm output của version trước để notebook tự tái sử dụng, không dựng Dense index hoặc tải model lại.